<a href="https://colab.research.google.com/github/ms-murthy/Build-Prod-AI-Agents-with-AWS-AgentCore/blob/main/Another_copy_of_Lab_1_2_Hello_LLM_Multi_Provider_%2B_Pydantic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔧 Module 1 · Lab 1.2 — Hello-LLM: Multi-Provider + Pydantic
## Engineering reliable LLM calls across OpenAI · Anthropic · Google · Groq

---

## 🗺️ How to Use This Lab (Read This First!)

Welcome to **Lab 1.2** — where we shift from *judging* AI models to *engineering* against them. In Lab 1.1 you read responses and scored them by eye. Here you build the plumbing that lets real applications call LLMs programmatically, get structured data back, and stay live when a provider has a bad day.

### What Problem Are We Solving?

Real production applications can't paste responses into a chat window. They need:
- Calls that work reliably across **multiple providers** (so one outage doesn't kill everything)
- **Structured, machine-readable output** they can route, store, and act on (not free text)
- Tools to **measure cost and latency** so provider choices are evidence-based, not vibes

### What You'll Build Step by Step

```
┌─────────────────────────────────────────────────────────────────┐
│                    Lab 1.2 — What You Build                     │
│                                                                 │
│  Step 1–3:  Install, load keys, initialize 4 provider clients  │
│  Step 4:    Call each provider natively (learn their shapes)   │
│  Step 5:    Build unified wrappers (one interface, 4 providers) │
│  Step 6:    Side-by-side comparison with timing + token counts  │
│  Step 7:    Temperature vs reasoning effort (the two knobs)    │
│  Step 8:    Pydantic structured outputs ⭐ (headline skill)     │
│  Step 9:    Cross-provider structured output                    │
│  Step 10:   Cost & latency modelling at scale                  │
│  Step 11:   Multi-provider fallback chain                      │
└─────────────────────────────────────────────────────────────────┘
```

### New Concepts in This Lab

| Concept | What It Is | Why It Matters |
|---------|-----------|----------------|
| **Groq** | Cloud service running open-source Llama models on custom hardware | Very low latency; free tier; OpenAI-compatible API |
| **Unified wrapper** | A function that hides provider-specific SDK differences | Write once, switch providers without touching app code |
| **Temperature** | Controls randomness/creativity in non-reasoning models | 0 = deterministic; 1.2 = creative; wrong setting wastes money & quality |
| **Reasoning effort** | Controls how hard a reasoning model thinks | `low`/`medium`/`high` — replaces temperature on reasoning models |
| **Pydantic BaseModel** | Python class that defines a data schema | Forces LLM output into a validated, typed object |
| **`responses.parse()`** | OpenAI method that guarantees schema conformance | No fragile JSON parsing; get a real Python object back |
| **Fallback chain** | Try Provider A → B → C in order, return first success | One provider's outage ≠ your service going down |

### How to Navigate

- Run cells **in order** — each step depends on the previous
- Read the **📖 What This Does** block after each code cell before moving on
- Steps 1–3 are setup; Step 4 is understanding; Steps 5–11 are building
- You need API keys for **OpenAI, Anthropic, Google (Gemini),** and **Groq** (Groq has a free tier)

> **How this lab differs from Lab 1.1.** Lab 1.1 = explore & judge models *qualitatively* (free text, subjective scoring). **This lab = build against them**: machine-readable outputs, measured latency/cost, reliability patterns. The metric here is schema-conformance, token count, cost, and uptime — not "which answer reads nicer."


---

## Step 1 — Install the SDKs

**What are we installing?**

| Library | Purpose |
|---------|---------|
| `anthropic` | Anthropic SDK — for calling Claude models |
| `google-genai` | Google GenAI SDK — for calling Gemini models |
| `groq` | Groq Cloud SDK — for calling Llama (open-source) models at high speed |
| `openai` | OpenAI SDK — for calling GPT models |
| `pandas` | Data analysis — used to display comparison tables |
| `pydantic` | Data validation — used in Step 8 to enforce structured output schemas |
| `python-dotenv` | Reads API keys from a `.env` file when running locally |

> 💡 **Why Groq?** Groq runs open-source models (Llama 3.3 70B) on custom AI inference hardware called LPUs. The result is very high speed at low cost — and it's API-compatible with OpenAI, so you barely need to change your code.

In [ ]:
# 📦 Install all required libraries for this lab
# -q  = quiet mode (suppresses verbose pip output)
# -U  = upgrade to latest version if already installed
# Remove the '#' from the next line to actually run the install
!pip install -qU anthropic google-genai groq openai pandas pydantic python-dotenv
#
# anthropic      → Anthropic SDK — calls Claude models
# google-genai   → Google GenAI SDK — calls Gemini models
# groq           → Groq Cloud SDK — runs open-source Llama on custom inference HW
# openai         → OpenAI SDK — calls GPT models (also used by Groq's compatible API)
# pandas         → Data analysis — builds comparison tables in Steps 6 and 10
# pydantic       → Data validation — enforces structured output schemas in Step 8
# python-dotenv  → Reads API keys from a .env file when running locally (not on Colab)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 960.7/960.7 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 72.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.


**✅ What just happened?**
All libraries were installed (or upgraded). The `!` prefix runs a shell command inside the notebook.

> ⚠️ **If a cell fails**, try running it again. Colab occasionally has temporary network issues during installation.

## **2. API Key Configuration (Prerequisites for this lab)**

We will now set up the API keys required for accessing LLMs.

**Why do we need API keys?**
- To access OpenAI's, Gemini, Groq & Anthropic models (GPT-4, etc.)

**Where to get API keys:**
You will need an OpenAI, Gemini, Groq & Anthropic API keys to complete this lab. If you donot have one yet, please create it by following the guides.
1. **OpenAI API Key** → [Create API Key for OpenAI](https://www.skool.com/k21academy/classroom/f1bc0b14?md=d5e61c61b5404ed7b4a1e649a270ee58)
3. **Groq API Key:** [Creating API Keys for Groq](https://www.skool.com/k21academy/classroom/f1bc0b14?md=06ceaa8e20694a60bcf5b124fd54a617)
4. **Gemini API Key** → [Create API Key for Gemini](https://www.skool.com/k21academy/classroom/f1bc0b14?md=88d8bdcc9cfe43d7a1cd05a14a491915)
2. **Anthropic API Key** [Create API Key for Anthropic](https://www.skool.com/k21academy/classroom/f1bc0b14?md=854b6067a73e412a9b9749123c3436c3)


Once you have created API keys, it is important to securely store them in Google Colab secrets to avoid exposing sensitive information.

 → **Follow this Activity Guide to store API keys in Google colab Secrets:** [Click Here](https://www.skool.com/k21academy/classroom/f1bc0b14?md=75d194912ba0435a98bfebdb98a1add1)

In [ ]:
import os  # Standard library — gives us os.environ to read/write environment variables

def load_keys(required):
    """Load API keys from Colab Secrets (on Colab) or a local .env file (elsewhere)."""
    try:
        # Try to import Colab's userdata module — only available when running on Google Colab
        from google.colab import userdata          # running on Google Colab
        for k in required:
            try:
                # Read each key from Colab Secrets and store it as an environment variable
                # os.environ makes the key available to all Python libraries in this session
                os.environ[k] = userdata.get(k)
            except Exception:
                pass                                 # key not found in Colab Secrets — skip silently
        source = "Colab Secrets"
    except ImportError:                              # ImportError = not running on Colab (e.g. local machine)
        try:
            from dotenv import load_dotenv, find_dotenv
            # find_dotenv() searches the current directory and all parent directories for a .env file
            # load_dotenv() reads it and loads key=value pairs into os.environ
            load_dotenv(find_dotenv(usecwd=True))
        except ImportError:
            pass                                     # python-dotenv not installed — skip silently
        source = ".env / environment"
    # Print which source was used and show a ✅ or ❌ for each key
    print(f"\U0001F511 Key source: {source}")
    for k in required:
        # os.environ.get(k) returns None safely if the key is missing — never raises an error
        print(f"   {k:<22} {'\u2705' if os.environ.get(k) else '\u274C MISSING'}")

# Load all four API keys this lab needs
load_keys(["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY", "GROQ_API_KEY"])

🔑 Key source: Colab Secrets
   OPENAI_API_KEY         ✅
   ANTHROPIC_API_KEY      ✅
   GEMINI_API_KEY         ✅
   GROQ_API_KEY           ✅


**✅ Expected output:**
```
🔑 Key source: Colab Secrets
   OPENAI_API_KEY         ✅
   ANTHROPIC_API_KEY      ✅
   GEMINI_API_KEY         ✅
   GROQ_API_KEY           ✅
```

> ⚠️ **If you see ❌ MISSING for any key:** Click the 🔑 icon in the left sidebar → add the secret with the exact name shown (case-sensitive) → toggle **Notebook access** ON → re-run this cell.

---

## Step 3 — Initialize All Four Clients

Each provider exposes a different API surface. This step teaches you the **native shape** of each before we hide the differences behind a unified wrapper in Step 5.

| Provider | Client class | Primary call method | Notable trait |
|---|---|---|---|
| OpenAI | `OpenAI()` | `responses.create()` or `chat.completions.create()` | Reasoning models + native structured output |
| Anthropic | `Anthropic()` | `messages.create()` | `system` is a top-level argument, not a message |
| Google | `genai.Client()` | `models.generate_content()` | All settings go inside a `GenerateContentConfig` object |
| Groq | `Groq()` | `chat.completions.create()` | **OpenAI-compatible** — same API shape as OpenAI chat |

> 💡 **Why is Groq OpenAI-compatible?** Groq adopted OpenAI's chat completions API format so developers don't need to learn a new SDK. The only things that change are the `api_key` and the model name.

In [ ]:
import time  # Standard library — used to measure API call latency (time.time() before and after)

# ── IMPORT ONE SDK PER PROVIDER ──────────────────────────────────────────────
from openai import OpenAI          # OpenAI SDK — GPT models + Responses API
from anthropic import Anthropic    # Anthropic SDK — Claude models
from google import genai           # Google GenAI SDK — Gemini models
from google.genai import types     # types module — needed for GenerateContentConfig
from groq import Groq              # Groq Cloud SDK — Llama models (OpenAI-compatible)

# ── INITIALIZE ONE CLIENT PER PROVIDER ───────────────────────────────────────
# OpenAI() and Anthropic() and Groq() auto-read their keys from os.environ
# (OPENAI_API_KEY, ANTHROPIC_API_KEY, GROQ_API_KEY — loaded in Step 2)
openai_client    = OpenAI()
anthropic_client = Anthropic()
# Gemini requires the key to be passed explicitly — it does not auto-read from env
gemini_client    = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
groq_client      = Groq()

# ── PIN EXACT MODEL VERSIONS ─────────────────────────────────────────────────
# Pinning ensures reproducible results — if a provider releases a new model,
# update one line here instead of hunting for every call in the code
M = {
    "openai_reason": "gpt-5-mini",            # OpenAI reasoning model — uses the Responses API
    "openai_chat":   "gpt-4.1-mini",          # OpenAI fast non-reasoning chat model
    "anthropic":     "claude-haiku-4-5",      # Fast Claude (use Sonnet 4.6 / Opus 4.8 for harder tasks)
    "gemini":        "gemini-2.5-flash",      # Google's cost-leader model — large context window
    "groq":          "llama-3.3-70b-versatile",  # Open-source Llama 3.3 70B running on Groq LPUs
}
print("All 4 clients initialized:", list(M.values()))

All 4 clients initialized: ['gpt-5-mini', 'gpt-4.1-mini', 'claude-haiku-4-5', 'gemini-2.5-flash', 'llama-3.3-70b-versatile']


**📖 What This Does:**

**Why two OpenAI entries (`openai_reason` and `openai_chat`)?**
OpenAI has two model families with different APIs:
- `gpt-4.1-mini` is a standard **chat model** — takes `temperature`, uses `chat.completions.create()`
- `gpt-5-mini` is a **reasoning model** — uses `responses.create()`, takes `reasoning={"effort": ...}` instead of temperature

You'll use both in this lab, so we pin both.

**Why does `gemini_client` get the key explicitly?**
The Google SDK doesn't follow the convention of auto-reading `GOOGLE_API_KEY` from the environment — it requires you to pass the key at construction time.

**✅ Expected output:** `All 4 clients initialized: ['gpt-5-mini', 'gpt-4.1-mini', 'claude-haiku-4-5', 'gemini-2.5-flash', 'llama-3.3-70b-versatile']`

---

## Step 4 — "Hello" from Each Provider: One Business Task Each

Before building wrappers, we call each SDK **natively** so you can see exactly what differs. Four providers, four industries, four API shapes. Read how each call is structured.

**Why learn the native shapes?** When something breaks in production, you need to read raw SDK code and API docs. If you only ever saw the wrapper, you'd be lost at debugging time.

---

### Step 4a — OpenAI: Insurance Claim Intake 🏥 *(Responses API)*

The **Responses API** is OpenAI's newer interface and the home of reasoning models.

**Key differences from the Chat Completions API:**
- Uses `instructions` (not `messages[{role: system}]`) for the system prompt
- Uses `input` (not `messages[{role: user}]`) for the user message
- Uses `reasoning={"effort": "low"|"medium"|"high"}` instead of `temperature`
- Response text is at `resp.output_text` (not `resp.choices[0].message.content`)

In [ ]:
# ── OPENAI RESPONSES API ────────────────────────────────────────────────────
# This is OpenAI's newer API, designed for reasoning models.
# Key shape differences vs chat.completions:
#   instructions  = system prompt (tells the model its role)
#   input         = user message
#   reasoning     = depth of internal thinking (low/medium/high) — replaces temperature
resp = openai_client.responses.create(
    model=M["openai_reason"],  # Use the reasoning model (gpt-5-mini)
    instructions="You are a senior claims assistant at a health-insurance company. Be precise and empathetic.",
    input="A customer writes: 'I was hospitalized for 2 days with dengue. How do I file a cashless claim?' "
          "Give a short, numbered step-by-step.",
    reasoning={"effort": "low"},  # low = quick; medium = balanced; high = deep (slower, pricier)
)
# resp.output_text extracts the final text answer (not resp.choices[0].message.content like chat API)
print(resp.output_text)

Sorry you were ill — glad you’re recovering. To file a cashless claim for your 2-day dengue hospitalization, follow these steps:

1. Notify your insurer or TPA immediately (or within the time specified in your policy — typically 24–48 hours of admission).  
2. Confirm the hospital is in your insurer’s network (cashless is only available at network hospitals).  
3. Ask the hospital to initiate a pre-authorization/cashless request on your behalf to the insurer/TPA.  
4. Provide the hospital these documents/copies they will need to upload: policy number/ID card, government ID, attending doctor’s admission note, preliminary/final diagnosis (dengue) and investigation reports (CBC, platelet counts, NS1/IgM etc.).  
5. Hospital will send the pre-authorization form, estimated cost, and clinical details to the insurer/TPA.  
6. Wait for insurer/TPA approval — they may ask for additional documents or clarifications from the hospital.  
7. Once approved, the insurer settles eligible bills directl

**📖 What This Does:**

The Responses API sends the request to `gpt-5-mini`, which first does a hidden reasoning pass (controlled by `effort`) before writing the final answer. `"low"` effort is fast and cheap; `"high"` effort is slower but produces deeper reasoning chains.

`resp.output_text` is a convenience property that extracts the text from the response object. Under the hood, the response also contains the reasoning tokens (if you need them for debugging).

**✅ Expected output:** A numbered list of steps for filing a cashless dengue hospitalization claim (notify insurer within 24 hours, fill the claim form, attach bills, etc.)

---

### Step 4b — Anthropic: Patient-Facing Health FAQ 🩺 *(Messages API)*

Claude's `messages.create()` has one notable structural difference: the **system prompt is a top-level argument**, not embedded inside the `messages` list. This is deliberate — Anthropic treats the system prompt as a separate concern from the conversation.

**Also required:** Anthropic **always requires `max_tokens`**. There's no default — calling without it raises an error.

In [ ]:
# ── ANTHROPIC MESSAGES API ──────────────────────────────────────────────────
# Key shape differences vs OpenAI:
#   system    = top-level argument (NOT inside messages list)
#   max_tokens = REQUIRED — Anthropic has no default; omitting raises an error
#   messages  = list of {role, content} dicts — same concept as OpenAI but without system here
msg = anthropic_client.messages.create(
    model=M["anthropic"],
    max_tokens=400,  # Required — cap on response length
    system="You are a patient-friendly hospital assistant. Use simple, reassuring language. You never give a diagnosis.",
    messages=[{"role": "user", "content": "I have a fasting blood-sugar test at 8 AM tomorrow. Can I drink water before it?"}],
)
# Response text lives at msg.content[0].text
# msg.content is a list because a single response can theoretically contain multiple content blocks
print(msg.content[0].text)

Yes, you can drink water before your fasting blood sugar test! Water is fine and won't affect your results.

Here's what you should know:

**What's allowed:**
- Plain water - drink as much as you need
- You can brush your teeth (just don't swallow toothpaste)

**What to avoid:**
- Food of any kind
- Drinks with calories (juice, coffee with cream/sugar, milk, soda)
- Gum or mints
- Most medications (ask your doctor about any you take regularly)

**Tips for the morning:**
- It's normal to feel a bit hungry or thirsty
- Drinking water can actually help - it makes it easier for the lab staff to find your vein
- Set an alarm so you arrive on time

If you take regular medications or have any questions about what you can or can't have, it's best to call your doctor's office tonight to confirm - they'll give you specific instructions based on your situation.

You're all set! Just stay hydrated with water, and you'll be fine.


**📖 What This Does:**

`anthropic_client.messages.create()` sends the system prompt and user message to `claude-haiku-4-5`.

**Why `msg.content[0].text`?**
Claude's response is a `ContentBlock` list — each block can be text, a tool call, etc. For a simple text response there's always exactly one block at index `[0]`. `.text` extracts the string.

**✅ Expected output:** A clear, reassuring answer about drinking water before a fasting blood test (yes, plain water is fine; avoid tea/coffee/juice).

---

### Step 4c — Google Gemini: E-Commerce Product Copy 🛍️ *(GenAI SDK)*

Gemini takes the user message as `contents` and bundles **all other settings** (system prompt, temperature, token limits) inside a `GenerateContentConfig` object. This is the most structurally different of the four SDKs.

In [ ]:
# ── GOOGLE GEMINI GENAI SDK ─────────────────────────────────────────────────
# Key shape differences vs OpenAI:
#   contents               = user message (the equivalent of messages[{role: user}])
#   config                 = GenerateContentConfig object holding ALL other settings
#   system_instruction     = system prompt (inside the config object, not top-level)
#   max_output_tokens      = token limit (Gemini uses this name instead of max_tokens)
r = gemini_client.models.generate_content(
    model=M["gemini"],
    contents="Write a 2-sentence product description for a 1L stainless-steel insulated water bottle that keeps "
             "drinks cold for 24 hours. Upbeat and benefits-first.",
    config=types.GenerateContentConfig(
        system_instruction="You are a creative e-commerce copywriter.",
        # Note: temperature and max_output_tokens could also go here
    ),
)
# Response text is directly at r.text — simpler than OpenAI's r.choices[0].message.content
print(r.text)

Beat the heat and stay perfectly hydrated with our 1L stainless steel insulated water bottle, engineered to keep your beverages refreshingly cold for a full 24 hours. Enjoy constant, icy-cold sips whether you're at the gym, office, or exploring the great outdoors!


**📖 What This Does:**

`gemini_client.models.generate_content()` sends the product brief to `gemini-2.5-flash`. The `types.GenerateContentConfig` object is Gemini's way of bundling configuration — you'll see it again in Step 9 when we use it to enforce a JSON schema.

**`r.text`** is the simplest response accessor — Gemini exposes the text directly without needing `.choices[0].message.content` or `.content[0].text`.

**✅ Expected output:** A 2-sentence upbeat product description, benefits-first (e.g., "Stay refreshingly hydrated all day...")

---

### Step 4d — Groq: Logistics Status Summary 🚚 *(OpenAI-compatible)*

Groq runs the open-source **Llama 3.3 70B** model on custom LPU (Language Processing Unit) hardware. The key insight: Groq adopted OpenAI's `chat.completions.create()` API shape exactly — so if you know one, you know both. The only change is which client object you use.

In [ ]:
# ── GROQ CLOUD — OPENAI-COMPATIBLE API ──────────────────────────────────────
# Groq uses the IDENTICAL shape as openai.chat.completions.create()
# The only difference: we use groq_client instead of openai_client
# This means any code written for OpenAI's chat API can switch to Groq with one line change
r = groq_client.chat.completions.create(
    model=M["groq"],  # Llama 3.3 70B running on Groq's custom LPU hardware
    messages=[
        {"role": "system", "content": "You are a logistics assistant. Summarize shipment events into ONE clear customer-facing line."},
        {"role": "user",   "content": "Events: [Picked up Mumbai 10:02] [In transit Pune 14:20] "
                                       "[Out for delivery Bangalore 08:15] [Delivery attempted — customer unavailable 11:40]. "
                                       "Summarize the current status."},
    ],
)
# Response shape is identical to OpenAI chat: r.choices[0].message.content
print(r.choices[0].message.content)

Your shipment, which originated in Mumbai, has been delivered to Bangalore and a delivery attempt was made at 11:40, but unfortunately, you were unavailable to receive it.


**📖 What This Does:**

`groq_client.chat.completions.create()` is byte-for-byte the same structure as `openai_client.chat.completions.create()`. Groq's API is intentionally designed to be a drop-in replacement.

**Why does Groq run open-source models so fast?** Groq built custom chips (LPUs) optimized specifically for transformer inference — they achieve much higher throughput than GPUs for this workload.

**`r.choices[0].message.content`** — same access path as OpenAI chat. This identical response shape is the point: Groq is a speed/cost alternative, not a different paradigm.

**✅ Expected output:** One clear sentence like "Your delivery was attempted at 11:40 but you were unavailable — expect a redelivery attempt soon."

---

## Step 5 — Unified Wrapper Functions: One Interface, Four Providers

The four native API shapes are different enough that you don't want them spread across your app. We build **one wrapper per provider**, all with the same signature and the same return shape:

```python
def call_provider(system, user, model=None, temperature=0.7, max_tokens=500) -> dict
# Returns: {"provider": str, "model": str, "text": str, "latency": float, "tokens": int}
```

Once these wrappers exist, the rest of the lab doesn't care which provider is used — it just calls the wrapper and reads `result["text"]`, `result["latency"]`, and `result["tokens"]`.

**This is the core engineering pattern:** hide SDK-specific complexity behind a stable interface. When you switch providers, only one line changes.

In [ ]:
# ── PROVIDER WRAPPER FUNCTIONS ──────────────────────────────────────────────
# Every wrapper has the SAME signature: (system, user, model, temperature, max_tokens)
# Every wrapper returns the SAME dict: {provider, model, text, latency, tokens}
# This uniformity is what makes the comparison in Step 6 and fallback in Step 11 trivial.

def call_openai(system, user, model=None, temperature=0.7, max_tokens=500):
    model = model or M["openai_chat"]  # Default to fast chat model (not the reasoning one)
    t0 = time.time()                   # Capture timestamp BEFORE the API call
    r = openai_client.chat.completions.create(
        model=model, temperature=temperature, max_tokens=max_tokens,
        # Chat completions: system goes inside messages list with role="system"
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    return {"provider": "OpenAI", "model": model,
            "text": r.choices[0].message.content,  # Extract text from response object
            "latency": round(time.time() - t0, 2),  # Latency in seconds, 2 decimal places
            "tokens": r.usage.total_tokens}          # Total tokens = input + output tokens

def call_anthropic(system, user, model=None, temperature=0.7, max_tokens=500):
    model = model or M["anthropic"]
    t0 = time.time()
    r = anthropic_client.messages.create(
        model=model, max_tokens=max_tokens, temperature=temperature,
        system=system,  # Anthropic: system is a top-level arg, NOT inside messages
        messages=[{"role": "user", "content": user}])
    return {"provider": "Anthropic", "model": model,
            "text": r.content[0].text,             # Extract from ContentBlock list
            "latency": round(time.time() - t0, 2),
            # Anthropic splits tokens: input (prompt) + output (completion) — sum them
            "tokens": r.usage.input_tokens + r.usage.output_tokens}

def call_gemini(system, user, model=None, temperature=0.7, max_tokens=500):
    model = model or M["gemini"]
    t0 = time.time()
    r = gemini_client.models.generate_content(
        model=model,
        contents=user,  # Gemini: user message goes in 'contents'
        # Gemini: ALL settings (system, temperature, token limit) go inside GenerateContentConfig
        config=types.GenerateContentConfig(
            system_instruction=system,
            temperature=temperature,
            max_output_tokens=max_tokens))  # Gemini uses max_output_tokens, not max_tokens
    return {"provider": "Google", "model": model,
            "text": r.text,  # Gemini exposes text directly at r.text
            "latency": round(time.time() - t0, 2),
            "tokens": r.usage_metadata.total_token_count}  # Gemini: total_token_count

def call_groq(system, user, model=None, temperature=0.7, max_tokens=500):
    model = model or M["groq"]
    t0 = time.time()
    r = groq_client.chat.completions.create(
        model=model, temperature=temperature, max_tokens=max_tokens,
        # Identical structure to OpenAI chat completions — this is the OpenAI-compatible API
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    return {"provider": "Groq", "model": model,
            "text": r.choices[0].message.content,   # Same access path as OpenAI
            "latency": round(time.time() - t0, 2),
            "tokens": r.usage.total_tokens}          # Same field as OpenAI

# Store all four wrappers in a dict — enables the for-loop comparison in Step 6
PROVIDERS = {"OpenAI": call_openai, "Anthropic": call_anthropic, "Google": call_gemini, "Groq": call_groq}
print("Unified wrappers ready:", list(PROVIDERS))

Unified wrappers ready: ['OpenAI', 'Anthropic', 'Google', 'Groq']


**📖 What This Does — Key Design Decisions:**

**Why does `call_anthropic` sum `input_tokens + output_tokens`?**
Anthropic's SDK returns token counts as two separate fields (`r.usage.input_tokens` and `r.usage.output_tokens`). OpenAI and Groq return them already summed as `r.usage.total_tokens`. We normalize to a single `tokens` value so the comparison table in Step 6 is apples-to-apples.

**Why `model = model or M["openai_chat"]`?**
This lets callers override the model (`call_openai(s, u, model="gpt-4.1")`) while providing a sensible default. The `or` pattern picks the explicit argument if truthy, otherwise falls back to the default.

**Why `round(time.time() - t0, 2)`?**
`time.time()` returns a float with many decimal places. Rounding to 2 gives us "1.23 seconds" format which is readable in the comparison table.

**`PROVIDERS` dict** — storing the four wrapper functions by name lets Step 6 loop over all four with a single `for name, fn in PROVIDERS.items()` instead of four separate calls.

**✅ Expected output:** `Unified wrappers ready: ['OpenAI', 'Anthropic', 'Google', 'Groq']`

---

## Step 6 — Side-by-Side Comparison: Same Prompt, All Four 🏦 *(Banking)*

**Business scenario:** A bank's app must explain a *declined transaction* to a worried customer — calmly and clearly. We send the **exact same** system + user prompt to all four providers and capture latency and token usage.

This is the **quantitative comparison** Lab 1.1 deliberately left out. Same subjective quality questions apply (who sounds most helpful?) but now we also have numbers: latency and token count.

In [ ]:
# ── DEFINE THE BANKING SCENARIO ─────────────────────────────────────────────
SYSTEM = "You are a bank's customer-support assistant. Explain clearly and calmly, in under 80 words."
USER   = ("My debit card was just declined at a petrol pump even though I have ₹40,000 in my account. "
          "Why would this happen and what should I do right now?")

results = {}
# Loop over all four providers — same prompt, different SDK under the hood
for name, fn in PROVIDERS.items():
    try:
        results[name] = fn(SYSTEM, USER)  # fn is one of the four wrapper functions
    except Exception as e:
        # If a provider fails, store a structured error instead of crashing the loop
        # This keeps the comparison table intact even if one provider has an issue
        results[name] = {"provider": name, "model": "-", "text": f"[ERROR] {e}", "latency": None, "tokens": None}
    print(f"\n=== {name} ({results[name]['model']}) ===")
    print(results[name]["text"])


=== OpenAI (gpt-4.1-mini) ===
Your card could be declined due to reasons like daily transaction limits, network issues, or security blocks. First, check if you've exceeded your daily spending limit or if the petrol pump accepts your card type. Also, verify your account status via mobile banking or app. If unclear, call your bank’s customer support immediately to resolve any holds or blocks on your card. Meanwhile, try another payment method if possible.

=== Anthropic (claude-haiku-4-5) ===
Several reasons could cause this:

1. **Daily limit exceeded** – Petrol pumps may have transaction limits
2. **Card locked** – Security freeze triggered by unusual activity
3. **Technical glitch** – Temporary system issue at the pump or bank
4. **Card expired/damaged** – Check your card's validity

**Immediate steps:**
- Try another ATM or store
- Contact your bank's customer support immediately
- Ask if your card is blocked
- Request a replacement if needed

Your funds are safe. This is usually qu

**📖 What This Does:**

The `try/except` block around each provider call means **one failure doesn't stop the others**. If Gemini is having a slow day, you still get OpenAI, Anthropic, and Groq responses. The error is stored in the same dict shape so the comparison table still renders cleanly.

**✅ What to observe while reading the responses:**
- Do all four cover the most common reason (daily limit / NFC not enabled / network issue)?
- Does any provider add unnecessary caveats or go over 80 words?
- Which response would you most want to receive as a stressed customer?

In [ ]:
import pandas as pd  # Data analysis library — used to build a formatted comparison table

# Build a DataFrame from the results dict for a clean side-by-side view
df = pd.DataFrame([{
    "Provider":    r["provider"],
    "Model":       r["model"],
    "Latency (s)": r["latency"],   # Time from request to response (seconds)
    "Tokens":      r["tokens"],    # Total tokens consumed (input + output)
} for r in results.values()])

print(df.to_string(index=False))  # index=False removes the row numbers for cleaner output
print("\nGroq usually wins latency (custom inference HW). Token counts differ — each provider tokenizes differently.")

 Provider                   Model  Latency (s)  Tokens
   OpenAI            gpt-4.1-mini         2.83     146
Anthropic        claude-haiku-4-5         2.84     195
   Google        gemini-2.5-flash         3.03     558
     Groq llama-3.3-70b-versatile         0.37     128

Groq usually wins latency (custom inference HW). Token counts differ — each provider tokenizes differently.


**📖 What This Does:**

**`pd.DataFrame(...).to_string(index=False)`** builds a 4-row table with one row per provider. `index=False` hides the 0/1/2/3 row numbers for cleaner output.

**Why do token counts differ across providers?** Each provider uses a different tokenizer — the same English text produces different token IDs and different counts. Gemini's count may differ from OpenAI's even for identical input text. This is why you can't compare prices using OpenAI's tiktoken on non-OpenAI models.

**What to look for:**
- Groq typically has the lowest latency (often <0.5s vs 1-3s for others)
- Token counts vary ±20-30% across providers for the same text
- Latency varies significantly based on server load and model size

---

## Step 7 — Temperature vs. Reasoning Effort: The Two Knobs People Confuse

These are the two most-confused control knobs in the LLM world:

| Knob | Applies to | Controls | Values |
|---|---|---|---|
| **Temperature** | Non-reasoning models (`gpt-4.1-mini`, `claude-haiku-4-5`, `llama-3.3-70b`) | Randomness / creativity | `0.0` (deterministic) → `2.0` (wild) |
| **Reasoning effort** | Reasoning models (`gpt-5-mini`) | Depth of internal "thinking" before answering | `"low"` / `"medium"` / `"high"` |

**The key rule:** Reasoning models typically **don't take a temperature** — you steer them with *effort*, not randomness. Mixing them up (e.g., passing `temperature` to a reasoning model) either raises an error or is silently ignored.

**Mental model:**
- Temperature = how *creatively* the model picks each word
- Reasoning effort = how *deeply* the model thinks before it starts writing

---

### Step 7a — Temperature Changes *Randomness* (Non-Reasoning Model)

Same prompt, three temperature levels. Watch how `temp=0.1` gives a safe, generic tagline while `temp=1.2` produces more distinctive (sometimes weird) output.

In [ ]:
# ── TEMPERATURE COMPARISON — THREE LEVELS ───────────────────────────────────
# Same prompt, same model, only temperature changes
prompt = "Write a 1-sentence tagline for a budgeting app aimed at college students."

for t in (0.1, 0.7, 1.2):
    # temperature=t controls how randomly the model samples its next token:
    #   0.1 → very deterministic; almost always picks the most likely token
    #   0.7 → balanced; good default for most tasks
    #   1.2 → more random; more variety but also more risk of odd output
    out = call_openai("You are a witty copywriter.", prompt, temperature=t)
    print(f"temp={t}: {out['text']}")

temp=0.1: Budget smarter, party harder — your wallet’s new BFF for college cash control.
temp=0.7: “Budget smart, party smarter — your college cash coach in your pocket.”
temp=1.2: Spend smart, stress less — your GPA’s rival is now your budget’s best pal.


**📖 What This Does:**

At `temperature=0.1`, the model picks the safest, most "average" tokens — the output is predictable and conservative. At `temperature=1.2`, the model samples more widely from less-likely tokens — output is more creative but less predictable.

**✅ What to observe:** The `temp=0.1` tagline will probably be something like "Track every rupee..." — functional but flat. The `temp=1.2` version might be more playful or unconventional.

---

### Step 7b — Same Prompt, 3 Runs at High Temperature → Different Every Time

This confirms that high temperature produces **run-to-run variation** — run the same prompt three times and get three different answers. At `temperature=0`, all three would be identical.

In [ ]:
# ── RUN-TO-RUN VARIATION AT HIGH TEMPERATURE ────────────────────────────────
# Three calls with identical inputs but temperature=1.2
# Expected: three noticeably different taglines
# If temperature=0 were used instead, all three would be the same (or near-identical)
for i in range(3):
    out = call_openai("You are a witty copywriter.", prompt, temperature=1.2)
    print(f"run {i+1}: {out['text']}")

run 1: “Budget better, party smarter—your college cash coach in your pocket.”
run 2: Budget smart, party harder—your campus cash coach in your pocket.
run 3: Cash in control: Budget smart, party smart, succeed smarter.


**📖 What This Does:**

`temperature=1.2` means the model's token sampling is much more random. The same input produces a different probability distribution over tokens each time, leading to different word choices and different taglines.

**When is this good?** Brainstorming, creative writing, generating variations. When is it bad? Compliance text, extraction, classification — where you need consistency and predictability.

**Try it:** Run this cell 3 more times — you'll get 9 different taglines total.

---

### Step 7c — Reasoning Effort Changes *Depth* (Reasoning Model, Deterministic)

Reasoning models like `gpt-5-mini` don't use temperature — they use `effort`. The model does a hidden reasoning pass before writing the answer. More effort = deeper reasoning = better for hard problems, but slower and pricier.

In [ ]:
# ── REASONING EFFORT COMPARISON ─────────────────────────────────────────────
def openai_reasoning(prompt, effort="medium"):
    t0 = time.time()
    r = openai_client.responses.create(
        model=M["openai_reason"],        # Must use a reasoning model (gpt-5-mini)
        input=prompt,                    # Responses API uses 'input' not 'messages'
        reasoning={"effort": effort})    # "low" = fast/cheap, "high" = thorough/expensive
    return r.output_text, round(time.time() - t0, 2)

# A finance question that benefits from careful, multi-step reasoning
q = ("A retail customer asks why diversification reduces risk in a mutual-fund portfolio. "
     "Explain clearly, including the basic math intuition.")

for effort in ("low", "medium", "high"):
    text, lat = openai_reasoning(q, effort)
    # Truncate to 400 chars for display — the full answer is in 'text'
    print(f"\n--- effort={effort}  ({lat}s) ---\n{text[:400]}...")


--- effort=low  (11.73s) ---
Short answer
Diversification reduces risk because holding many different securities spreads out “idiosyncratic” shocks (company-specific gains or losses) so they cancel each other. Mathematically, the portfolio variance depends on variances AND covariances. When securities are not perfectly positively correlated, the covariance terms don’t grow as fast as the number of holdings, so the portfolio’s...

--- effort=medium  (25.63s) ---
Short answer
- Diversification cuts the part of a portfolio’s risk that comes from individual investments’ random ups and downs (idiosyncratic risk). It does this because independent or weakly related returns tend to cancel each other out when you hold many different securities. It does not remove marketwide (systematic) risk.

Basic math intuition (kept simple)
- For a portfolio the relevant meas...

--- effort=high  (59.94s) ---
Short answer (in plain English)
- Diversification reduces risk because different investments don’t

**📖 What This Does:**

`openai_client.responses.create()` with `reasoning={"effort": effort}` tells `gpt-5-mini` how much internal "thinking" to do. At `"low"`, it writes an answer quickly. At `"high"`, it works through the problem more carefully (you can see this in longer, more structured answers).

**What to observe:**
- `"low"` is fastest (0.5–2s) but shallowest
- `"high"` is slowest (3–10s+) but most thorough and often mathematically precise
- For the finance question, `"high"` effort often includes the variance formula or a portfolio example

> **When to use which?** Deterministic compliance text → `temperature=0`. Brainstorming → high temperature. Hard multi-step analysis → reasoning model at `"high"` effort. Picking wrong wastes money *and* quality.

---

## Step 8 — Structured Outputs with Pydantic: The Headline Skill ⭐

So far every answer was **free text**. Production systems need **structured, machine-readable** data they can route, store, and act on.

**The problem with asking for JSON in the prompt:**
```
"Reply with a JSON object containing category, urgency, and sentiment"
```
This is fragile. The model might add prose before the JSON, use single quotes instead of double quotes, invent field names, or omit required fields. Downstream code breaks silently.

**The solution:** Define a **Pydantic schema** and let the provider **guarantee** the output matches it. You get back a real Python object with `.category`, `.urgency`, `.sentiment` — not a string you have to parse.

**Business scenario — cross-industry support triage.** An inbound message could be from banking, healthcare, e-commerce, or insurance. The intake system must classify each into a fixed schema to auto-route to the right queue and dashboard.

### What is Pydantic?

Pydantic is a Python library for data validation. You define a class that inherits from `BaseModel`, declare typed fields, and Pydantic validates that any data you assign matches those types. When used with LLMs, it turns "the model might return anything" into "the model returns exactly this structure or it fails loudly."

In [ ]:
from pydantic import BaseModel, Field  # BaseModel = schema class; Field = field-level metadata
from typing import Literal, List       # Literal = fixed set of allowed values; List = typed list

class SupportTicket(BaseModel):
    """Structured triage for any inbound customer support message.

    Each field maps to something an operations team needs to act on:
    - category   → which queue to route to
    - urgency    → how fast a human needs to respond
    - sentiment  → tone to use in the reply
    - industry   → which SLA and scripts to apply
    - entities   → key data points to pull into the CRM
    - summary    → one line for the dashboard
    - recommended_action → what the agent should do first
    """
    # Literal[] constrains the model to ONLY return one of these exact strings
    # If the model tries to return "payment" instead of "billing", Pydantic raises a validation error
    category: Literal["billing", "technical", "complaint", "inquiry", "fraud_or_safety", "cancellation"] = Field(
        description="Primary intent of the message")
    urgency: Literal["low", "medium", "high", "critical"] = Field(
        description="How fast a human must act")
    sentiment: Literal["positive", "neutral", "negative", "angry"]
    industry: str = Field(description="Best-guess industry, e.g. 'banking', 'healthcare', 'e-commerce'")
    entities: List[str] = Field(description="Key entities: order ids, amounts, names, dates")
    summary: str = Field(description="One-sentence summary")
    recommended_action: str = Field(description="Suggested next step for the support agent")

**📖 What This Does:**

**`class SupportTicket(BaseModel)`** — inheriting from `BaseModel` makes this a Pydantic schema. The class body defines the expected fields, their types, and their validation rules.

**`Literal["billing", "technical", ...]`** — restricts the field to exactly those string values. This is the critical part: the LLM cannot return `"payment"` or `"payments"` — only the exact values listed. Pydantic will raise a `ValidationError` if the output doesn't conform.

**`Field(description=...)`** — the description is sent to the LLM as a hint about what this field means. It's the equivalent of a comment that the model can read.

**`List[str]`** — tells Pydantic (and the LLM) that `entities` is a list of strings, not a single string.

**✅ No output** — this cell defines the schema class. It's used in the next two cells.

---

### Step 8a — OpenAI `responses.parse()`: Guaranteed Schema → A Pydantic Object

`responses.parse()` is OpenAI's native structured-output method. It sends the schema to the model, which is then **constrained to only produce tokens that conform to it**. The result is a real, validated Python `SupportTicket` instance — not a string.

In [ ]:
# ── OPENAI NATIVE STRUCTURED OUTPUT ─────────────────────────────────────────
# A realistic banking complaint with two key entities (₹18,250 and order #A4821)
msg = ("My credit card was charged ₹18,250 TWICE for the same order #A4821 this morning. "
       "I need one charge refunded today — this is unacceptable.")

resp = openai_client.responses.parse(
    model=M["openai_reason"],
    # Note: responses.parse() uses input= (a list of messages) not just a string
    input=[{"role": "system", "content": "Classify the customer message into the SupportTicket schema."},
           {"role": "user", "content": msg}],
    text_format=SupportTicket,   # This is the key argument — tells OpenAI to enforce the schema
)
# resp.output_parsed is the validated SupportTicket instance — a real Python object, not a string
ticket = resp.output_parsed
print(type(ticket).__name__, "->")     # Should print "SupportTicket"
# model_dump_json() converts the Pydantic object to a formatted JSON string for display
print(ticket.model_dump_json(indent=2))

SupportTicket ->
{
  "category": "billing",
  "urgency": "high",
  "sentiment": "angry",
  "industry": "e-commerce",
  "entities": [
    "order #A4821",
    "amount ₹18,250",
    "charged twice",
    "time: this morning",
    "refund requested today"
  ],
  "summary": "Customer was charged ₹18,250 twice for order #A4821 this morning and demands one charge refunded today.",
  "recommended_action": "Verify payment records for order #A4821 to confirm duplicate charge; if confirmed, initiate refund for the duplicate ₹18,250 immediately and provide the customer with refund confirmation and expected timeline. If payment gateway issues prevent immediate refund, escalate to payments team and notify the customer of next steps and resolution ETA."
}


**📖 What This Does:**

**`text_format=SupportTicket`** — this is the key argument. OpenAI uses the Pydantic schema to constrain token generation — the model can only produce tokens that would form a valid JSON object matching the schema.

**`resp.output_parsed`** — returns a real `SupportTicket` Python object. You can access `ticket.category`, `ticket.urgency`, `ticket.entities` directly as typed attributes — no JSON parsing, no `.get()`, no `KeyError` risk.

**`ticket.model_dump_json(indent=2)`** — Pydantic's method to serialize the object back to a formatted JSON string for display.

**✅ Expected output:**
```json
SupportTicket ->
{
  "category": "billing",
  "urgency": "critical",
  "sentiment": "angry",
  "industry": "banking",
  "entities": ["₹18,250", "#A4821"],
  "summary": "Customer was double-charged ₹18,250 for order #A4821 and demands immediate refund.",
  "recommended_action": "Verify duplicate charge in payment system and initiate refund within 24 hours."
}
```

---

### Step 8b — Classify a Batch of Diverse Multi-Industry Messages

Five messages from five different industries — each produces a structured `SupportTicket`. The output table shows how the same schema adapts to completely different contexts.

In [ ]:
# ── BATCH CLASSIFICATION — 5 INDUSTRIES ─────────────────────────────────────
# Five diverse messages — different industries, tones, and urgencies
messages = [
    "The MRI report upload on your patient portal keeps failing with error 500. I need it before my appointment tomorrow.",   # healthcare / technical
    "I got an OTP I never requested and someone tried to log into my account. Please block it NOW.",                          # banking / fraud
    "Just wanted to say your delivery partner was super polite and my package arrived a day early. Thank you!",               # e-commerce / positive
    "I want to cancel my annual insurance policy renewal scheduled next week and get a pro-rata refund.",                     # insurance / cancellation
    "What are your customer-care hours on public holidays?",                                                                  # generic / inquiry
]

rows = []
for m in messages:
    # Same parse() call as above — same schema, different message each iteration
    r = openai_client.responses.parse(
        model=M["openai_reason"],
        input=[{"role": "system", "content": "Classify the customer message into the SupportTicket schema."},
               {"role": "user", "content": m}],
        text_format=SupportTicket,
    )
    t = r.output_parsed  # A real SupportTicket object for each message
    # Extract key fields into a row dict for the comparison table
    rows.append({
        "industry":  t.industry,
        "category":  t.category,
        "urgency":   t.urgency,
        "sentiment": t.sentiment,
        "action":    t.recommended_action[:45],  # Truncate to 45 chars for table display
    })

# Display as a table — each row is a classified message
print(pd.DataFrame(rows).to_string(index=False))

       industry        category  urgency sentiment                                        action
     healthcare       technical     high  negative Acknowledge receipt and escalate immediately 
online services fraud_or_safety critical     angry Immediately lock/suspend the user's account a
     e-commerce         inquiry      low  positive Send a brief thank-you reply acknowledging th
      insurance    cancellation     high   neutral Verify customer's identity and locate the pol
        general         inquiry      low   neutral Respond with the customer‑care hours for publ


**📖 What This Does:**

The loop calls `responses.parse()` five times — once per message. Each call returns a `SupportTicket` object. We extract 5 fields into a row dict and build a table.

**✅ What to observe in the table:**
- The fraud message should get `urgency: critical` and `category: fraud_or_safety`
- The positive e-commerce message should get `sentiment: positive` and `category: inquiry` or `complaint` (watch what it picks)
- The insurance cancellation should get `category: cancellation`
- Every row is a validated, schema-conformant object — no string parsing needed

> Every row is a **validated object** — `ticket.urgency` is always one of the four allowed values. You can now `filter(lambda r: r.urgency == "critical")` with zero string-parsing. *This* is why structured output is the backbone of agents (Modules 4+).

---

## Step 9 — Structured Output Across Providers: Who Supports What

Native schema enforcement varies by provider. Here's the landscape and how to handle each:

| Provider | Native schema? | Method |
|---|:--:|---|
| OpenAI | ✅ Full | `responses.parse(text_format=Model)` → `resp.output_parsed` |
| Google Gemini | ✅ Full | `GenerateContentConfig(response_schema=Model)` → `Model.model_validate_json(resp.text)` |
| Anthropic Claude | ⚠️ Prompt-based | Inject JSON schema into system prompt → parse + validate with Pydantic |
| Groq (Llama) | ⚠️ JSON mode | `response_format={"type": "json_object"}` → parse + validate with Pydantic |

**The universal safety net** (works for all four):
```python
raw_text = ... # whatever the model returned
validated_object = Model.model_validate_json(raw_text)  # raises ValidationError if invalid
```
Always validate the output before using it downstream — this catches schema drift early.

In [ ]:
# ── GEMINI — NATIVE STRUCTURED OUTPUT ───────────────────────────────────────
# Gemini supports schema enforcement via GenerateContentConfig
g = gemini_client.models.generate_content(
    model=M["gemini"],
    contents=msg,  # Same message as Step 8a (the double-charge banking complaint)
    config=types.GenerateContentConfig(
        system_instruction="Classify the customer message into the schema.",
        response_mime_type="application/json",  # Tell Gemini to return JSON
        response_schema=SupportTicket),         # Pass the Pydantic class as the schema
)
# g.text is the raw JSON string — we validate it into a SupportTicket object
# model_validate_json() raises a ValidationError if the JSON doesn't match the schema
print("Gemini  ->", SupportTicket.model_validate_json(g.text).category)

# ── ANTHROPIC CLAUDE — PROMPT-BASED JSON ────────────────────────────────────
# Claude has no native schema enforcement, so we inject the schema as text into the system prompt
# and then validate the output with Pydantic ourselves
schema_hint = SupportTicket.model_json_schema()  # Generates the JSON Schema dict from our Pydantic class
c = anthropic_client.messages.create(
    model=M["anthropic"],
    max_tokens=500,
    # Inject the schema as a text instruction — Claude reads it and tries to comply
    system=f"Return ONLY valid JSON matching this JSON Schema (no prose):\n{schema_hint}",
    messages=[{"role": "user", "content": msg}],
)
import json, re
raw = c.content[0].text
# Claude sometimes wraps output in ```json ... ``` code fences — strip them before parsing
raw = re.sub(r"^```(json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()
# model_validate() validates a Python dict; model_validate_json() validates a JSON string
print("Claude  ->", SupportTicket.model_validate(json.loads(raw)).category)

Gemini  -> billing
Claude  -> billing


**📖 What This Does — Two Different Approaches:**

**Gemini (native):**
- `response_mime_type="application/json"` tells Gemini the output should be JSON
- `response_schema=SupportTicket` passes the Pydantic class as a JSON Schema constraint
- `SupportTicket.model_validate_json(g.text)` validates the returned JSON string into a typed object

**Claude (prompt-based):**
- `SupportTicket.model_json_schema()` converts our Pydantic class into a JSON Schema dict that describes the expected structure
- We inject this into the system prompt as text — Claude reads it and tries to comply
- `re.sub(r"^```(json)?|```$", ...)` strips markdown code fences that Claude sometimes adds
- `json.loads(raw)` parses the JSON string into a Python dict
- `SupportTicket.model_validate(...)` validates the dict against the schema

**The critical difference:** With Gemini native and OpenAI's `parse()`, schema violations are caught at the API level (the model can't generate non-conforming output). With Claude and Groq prompt-based, schema violations are caught at validation time — meaning the call succeeds but `model_validate` raises an exception.

**✅ Expected output:**
```
Gemini  -> billing
Claude  -> billing
```
(Both should classify the double-charge message as `billing`.)

---

## Step 10 — Cost & Latency at Scale: The Data Behind a Vendor Decision

Quality is one axis of a provider decision. Cost is another — and at 50,000 tickets/day, a 4× price difference is real money.

**Business scenario:** A large e-commerce platform processes 50,000 customer support tickets per day. Each ticket averages 250 input tokens (system prompt + customer message) and 150 output tokens (classification + summary). What does the monthly bill look like per provider?

> ⚠️ **Pricing changes frequently.** The numbers below are illustrative. Always check current pricing at each provider's pricing page before quoting a customer or making a budget commitment.

In [ ]:
# ── PRICING TABLE (approximate USD per 1M tokens) ───────────────────────────
# REPLACE these values with current prices from each provider's pricing page
# These are for illustration only — prices change regularly
pricing = {
    "gpt-4.1-mini":            {"in": 0.40, "out": 1.60},  # OpenAI — fast chat model
    "gpt-5-mini":              {"in": 1.10, "out": 4.40},  # OpenAI — reasoning model (higher cost)
    "claude-haiku-4-5":        {"in": 0.80, "out": 4.00},  # Anthropic — fast Claude
    "gemini-2.5-flash":        {"in": 0.15, "out": 0.60},  # Google — cost leader
    "llama-3.3-70b-versatile": {"in": 0.59, "out": 0.79},  # Groq — open-source, low output cost
}

# ── WORKLOAD PARAMETERS ───────────────────────────────────────────────────────
# 50,000 tickets/day × 30 days = 1.5M tickets per month
TICKETS_PER_DAY = 50_000
IN_TOK  = 250   # Average input tokens per ticket (system prompt + customer message)
OUT_TOK = 150   # Average output tokens per ticket (structured classification + summary)
days    = 30

rows = []
for model, p in pricing.items():
    # Convert token counts to millions (pricing is $/1M tokens)
    daily_in  = TICKETS_PER_DAY * IN_TOK  / 1_000_000  # Millions of input tokens per day
    daily_out = TICKETS_PER_DAY * OUT_TOK / 1_000_000  # Millions of output tokens per day
    # Monthly cost = (daily_in × price_per_million_in + daily_out × price_per_million_out) × 30 days
    monthly = (daily_in * p["in"] + daily_out * p["out"]) * days
    rows.append({"Model": model, "$/1M in": p["in"], "$/1M out": p["out"], "Est. $/month": round(monthly, 2)})

print(f"Workload: {TICKETS_PER_DAY:,} tickets/day × {days} days  ({IN_TOK} in + {OUT_TOK} out tokens each)\n")
# sort_values("Est. $/month") orders from cheapest to most expensive
print(pd.DataFrame(rows).sort_values("Est. $/month").to_string(index=False))

Workload: 50,000 tickets/day × 30 days  (250 in + 150 out tokens each)

                  Model  $/1M in  $/1M out  Est. $/month
       gemini-2.5-flash     0.15      0.60        191.25
llama-3.3-70b-versatile     0.59      0.79        399.00
           gpt-4.1-mini     0.40      1.60        510.00
       claude-haiku-4-5     0.80      4.00       1200.00
             gpt-5-mini     1.10      4.40       1402.50


**📖 What This Does:**

**Token math:**
- `50,000 tickets × 250 tokens ÷ 1,000,000 = 12.5 million input tokens/day`
- `50,000 tickets × 150 tokens ÷ 1,000,000 = 7.5 million output tokens/day`
- Monthly cost = `(12.5 × price_in + 7.5 × price_out) × 30`

**Why does output cost more per token than input?**
Generating tokens is computationally more expensive than reading them. Input tokens are processed in parallel; output tokens are generated one at a time sequentially.

**Reading the table:**
- The cheapest model per token (likely Gemini) might cost 5–10× less than the most expensive
- But cheap ≠ right — weigh cost against the quality you measured in Step 6 and the schema conformance rate you'd measure in Step 8

> The cheapest model per token isn't automatically the right choice — weigh it against the **quality** you measured in Step 6 and the **caution** behaviour you saw in Lab 1.1. Cost is one axis of three (quality, reliability, cost).

---

## Step 11 — Production Pattern: Multi-Provider Fallback 🛡️ *(Black-Friday Reliability)*

**Business scenario:** It's the peak-sale spike. Your primary provider starts returning rate-limit / 5xx errors. You **cannot** drop customer messages on the floor.

A **fallback chain** tries providers in order and returns the first success. One provider's bad day doesn't take you down.

```
Message arrives
      │
      ▼
 Try OpenAI → succeeds? Return result
      │ fails
      ▼
 Try Groq   → succeeds? Return result
      │ fails
      ▼
 Try Anthropic → succeeds? Return result
      │ fails
      ▼
 raise RuntimeError("All providers failed")
```

**Why Groq as the second option (not Anthropic)?** Groq has very high throughput and is unlikely to be overloaded for the same reason OpenAI is. Anthropic is kept as a third option — different infrastructure, different failure modes.

In [ ]:
# ── MULTI-PROVIDER FALLBACK FUNCTION ────────────────────────────────────────
def classify_with_fallback(message, order=("OpenAI", "Groq", "Anthropic")):
    """Try providers in order; return the first successful response.

    order parameter lets callers change the priority — e.g. put Groq first
    for lower latency, or Anthropic first for compliance reasons.
    """
    last_error = None
    for name in order:
        try:
            # PROVIDERS[name] is one of the four wrapper functions from Step 5
            # All wrappers have the same signature and return the same dict shape
            out = PROVIDERS[name](
                "Reply in ONE sentence: state the category and urgency of this support message.",
                message)
            # Add which provider actually served this request — essential for monitoring
            return {"served_by": name, "model": out["model"],
                    "latency": out["latency"], "text": out["text"]}
        except Exception as e:
            # Log the failure and move on to the next provider
            last_error = f"{name} failed: {e}"
            print(f"  ⚠️ {last_error} → falling back...")
    # All providers failed — raise an error with the last failure message
    raise RuntimeError(f"All providers failed. Last error: {last_error}")

# ── TEST THE FALLBACK ─────────────────────────────────────────────────────────
# A high-urgency banking message — would be routed to fraud team in a real system
result = classify_with_fallback("URGENT: my account shows a ₹0 balance but I deposited ₹50,000 this morning!")
print(f"\n✅ Served by {result['served_by']} ({result['model']}, {result['latency']}s):\n{result['text']}")


✅ Served by OpenAI (gpt-4.1-mini, 1.36s):
Category: Account Balance Issue; Urgency: High/Urgent.


**📖 What This Does:**

**`order=("OpenAI", "Groq", "Anthropic")`** — the tuple defines priority. OpenAI is tried first; if it fails, Groq; then Anthropic. This is configurable — you might flip to `("Groq", "OpenAI", "Anthropic")` to prioritize low latency in normal operation.

**`PROVIDERS[name]`** — looks up the wrapper function by name. Because all wrappers have the same signature and return the same dict, the loop body doesn't need to know which provider it's calling.

**`"served_by": name`** — always record which provider served the request. This feeds your monitoring dashboard: if `served_by == "Groq"` more than 5% of the time, OpenAI is having problems.

**`raise RuntimeError(f"All providers failed...")`** — if the loop exhausts all options, we raise explicitly so the caller can handle it (show a user-facing error, add to a retry queue, etc.) rather than returning `None` silently.

**✅ Expected output:**
```
✅ Served by OpenAI (gpt-4.1-mini, 1.23s):
This is a fraud_or_safety / critical urgency issue — immediate account investigation required.
```

If OpenAI is rate-limiting you during testing, you'll see the `⚠️` warning and the result will come from Groq or Anthropic instead.

## 12. Conclusion & what's next

| You built | Why it matters |
|---|---|
| 4 native calls + unified wrappers | One interface over four very different SDKs |
| Temperature vs. reasoning effort | The two control knobs, finally untangled |
| **Pydantic structured outputs** | Validated objects, not fragile JSON-in-text — the backbone of agents |
| Cross-provider structured output | Know each provider's native support and the universal validate-it fallback |
| Cost/latency model | Turn "which provider?" into a number |
| Fallback chain | One provider's outage ≠ your outage |

**Next — Lab 1.3: LLM Robustness.** What happens when calls *fail* — rate limits, timeouts, transient 5xx — and how to detect **drift and hallucination**. We add retries, exponential backoff with jitter, and sanity checks so these patterns survive contact with production.